# Train Duck chat on the A30
Fine-tune a pretrained 0.49B-parameter Qwen chat model with LoRA. The pilot uses authored agricultural dialogue to teach Duck how to ask for field details, explain missing records, and ground answers. Crop and NASA records belong in runtime retrieval. The held-out prompts measure reference-answer loss and provide examples for human review. This pilot does not establish field outcome accuracy.

[Qwen model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import hashlib
import json
import datetime

import torch
assert torch.cuda.is_available(), "Select a CUDA-enabled Jupyter kernel."
device = torch.device("cuda")
print(torch.cuda.get_device_name())
packages = {"transformers": "transformers==4.52.4", "peft": "peft==0.15.2"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
torch.manual_seed(42)
base_id = "Qwen/Qwen2.5-0.5B-Instruct"
run = Path("duck-chat-runs") / datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run.mkdir(parents=True)


## Training and held-out dialogue

In [ ]:
system = "You are Duck, an agricultural decision-support assistant. Answer plainly. Use provided plot evidence when available. Distinguish crop observations from recommendations. Never invent a crop, date, or measurement for a plot."
training = [["Hello","Hi, I’m Duck. I can help you get started with farming, compare crop rotations, or explain data for a field you select."],["How do I start a farm?","Start with your goal, available land, water and budget. For a field you already have, check drainage, get a soil test and review local planting windows. What would you like to grow?"],["What can I plant here?","I can use your selected plot to narrow the options. Tell me what grew there last season and whether you irrigate. Those details help compare crops and planting windows."],["Why does the last crop matter?","It can change residue, water use, available nitrogen, and pest or disease pressure. Check those effects before choosing the following crop."],["My field grew beans last year. What now?","Use beans as the previous crop in the plan. A crop from a different family may help break pest cycles. Confirm the locally suitable choices and any nitrogen credit with your farm records or local guidance."],["Can you guarantee the highest yield?","I can compare options and explain the evidence. Yield depends on local conditions, management and the season. Use local trial results and farm records to judge likely outcomes."],["How do I choose between rotation plans?","Compare planting windows, water demand, soil requirements, disease pressures, and your priorities. Look for measured outcomes from comparable fields or trials."],["What if the crop map is missing?","Enter known crops and years for the selected field. I can use that history to compare options and explain which details still need verification."],["What is the soil moisture right now?","I need a dated observation for the selected plot to report moisture. Satellite estimates can cover a larger area than your field, so use a field probe to check conditions before irrigating."],["Use NASA to choose my crop rotation","NASA observations can help explain weather and field conditions. I use agricultural evidence such as verified crop history, local trial results, and farmer priorities to compare rotations."],["How do I check for planting risk?","Check the crop’s local planting window, drainage, recent field conditions, and water supply. Use forecasts to check near-term weather risks and local agronomic guidance for crop choice."],["Is modeled soil pH enough?","Modeled pH gives an estimate for the sampled area and depth. Collect a representative soil sample before making a lime or fertilizer decision."],["Can I compare fields?","Select each plot and tell me your priorities. I can explain which evidence applies to each field and where coverage differs."],["Is a satellite crop map proof of a rotation?","Use crops dated at the same field across seasons and check that the field boundary overlaps. A single crop map can identify a candidate crop class for its mapped season."],["When should I sow?","Tell me the crop and whether the field is irrigated. I can compare the crop’s local planting window with the selected plot’s conditions."],["My data says woody wetlands. Can I rotate corn there?","Check the selected boundary against farm records and local land-use requirements. Wetland classifications and cultivated-field history need to be resolved before considering a crop sequence."],["What can I do for long-term soil health?","Compare ways to keep the soil covered, reduce erosion, diversify crop families, and manage nutrients. Choose options that fit your water supply and growing season, then track changes with repeated soil tests."],["Does a field label give me the yield?","Crop labels describe what was declared or mapped. To compare yields, use measured harvests from the field or comparable local trials."],["Hola, quiero empezar a cultivar","Hola, soy Duck. Empieza por revisar el suelo, el acceso al agua y las fechas locales de siembra. Si seleccionas una parcela, puedo ayudarte a comparar las opciones para ese terreno."],["What are your data sources?","I can explain the source, date, and footprint for retrieved records. For field decisions, start with local farm records and verified agricultural datasets."]]
held_out = [["I’m new to growing crops. Where do I begin?","Start with your goals, land, water and soil. Check local planting dates and choose crops appropriate to your experience."],["Nothing shows up for last season on my plot.","Enter the crop and year from your field records, and check that the drawn boundary matches the cultivated area."],["The map gives soil moisture. Will you use it to pick my crop sequence?","Soil moisture describes field conditions. I compare crop sequences using agricultural evidence such as crop records and trials."],["Will the recommended sequence definitely increase my harvest?","Compare measured yields from similar fields or local trials. Results depend on management, growing conditions and season."],["I see a crop classification for one season. How can I check if the rotation repeats?","Find verified crop labels for the same field across consecutive and repeated seasons. One year does not establish recurrence."]]
assert not {q.casefold().strip() for q, _ in training} & {q.casefold().strip() for q, _ in held_out}
dataset = {"origin": "authored Duck instruction examples", "training": training, "held_out": held_out}
payload = json.dumps(dataset, ensure_ascii=False, indent=2)
(run / "examples.json").write_text(payload, encoding="utf-8")
print(f"{len(training)} training examples and {len(held_out)} held-out prompts")


## Train and save the adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_id)
model = AutoModelForCausalLM.from_pretrained(base_id).to(device, dtype=torch.bfloat16)
revision = getattr(model.config, "_commit_hash", None)
base_parameters = sum(p.numel() for p in model.parameters())
assert base_parameters <= 1_000_000_000
model = get_peft_model(model, LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
))
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_parameters <= 1_000_000_000
print(f"{total_parameters:,} total parameters, {trainable_parameters:,} trainable")

def prompt(question):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True,
    )

def encode(question, answer):
    prefix = tokenizer.encode(prompt(question), add_special_tokens=False)
    completion = tokenizer.encode(answer, add_special_tokens=False) + [tokenizer.eos_token_id]
    tokens = torch.tensor([prefix + completion], device=device)
    labels = torch.tensor([[-100] * len(prefix) + completion], device=device)
    return tokens, labels

@torch.inference_mode()
def score(rows):
    model.eval()
    losses = []
    for question, answer in rows:
        tokens, labels = encode(question, answer)
        losses.append(model(input_ids=tokens, labels=labels, use_cache=False).loss.item())
    return sum(losses) / len(losses)

@torch.inference_mode()
def reply(question):
    model.eval()
    inputs = tokenizer(prompt(question), return_tensors="pt", add_special_tokens=False).to(device)
    output = model.generate(**inputs, max_new_tokens=110, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

with model.disable_adapter():
    baseline_loss = score(held_out)
    baseline_answers = [reply(q) for q, _ in held_out]

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=2e-4)
loss_history = []
completed_history = []
for epoch in range(3):
    model.train()
    losses = []
    for index in torch.randperm(len(training)).tolist():
        tokens, labels = encode(*training[index])
        optimizer.zero_grad(set_to_none=True)
        loss = model(input_ids=tokens, labels=labels, use_cache=False).loss
        if not torch.isfinite(loss):
            raise RuntimeError("Training loss became nonfinite.")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    loss_history.append(sum(losses) / len(losses))
    completed_history.append(len(losses))
    print(f"Epoch {epoch + 1}: {len(losses)}/{len(training)} examples completed", flush=True)

model.save_pretrained(run / "adapter", safe_serialization=True)
tokenizer.save_pretrained(run / "adapter")
assert (run / "adapter" / "adapter_model.safetensors").stat().st_size > 0


## Check held-out answers

In [ ]:
final_loss = score(held_out)
answers = [
    {"prompt": question, "reference": reference, "base": before, "fine_tuned": reply(question)}
    for (question, reference), before in zip(held_out, baseline_answers)
]
summary = {
    "base_model": base_id, "base_revision": revision, "gpu": torch.cuda.get_device_name(),
    "base_parameters": base_parameters, "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters, "training_count": len(training),
    "held_out_count": len(held_out), "examples_sha256": hashlib.sha256(payload.encode()).hexdigest(),
    "mean_training_loss_by_epoch": loss_history,
    "examples_completed_by_epoch": completed_history,
    "optimizer_steps_completed": sum(completed_history),
    "held_out_reference_loss_base": baseline_loss, "held_out_reference_loss_adapter": final_loss,
    "answers": answers,
}
(run / "training-report.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved adapter and report to {run.resolve()}")
print(f"Completed {sum(completed_history)} training steps and evaluated {len(answers)} held-out prompts.")
for item in answers:
    print("\n", item["prompt"], "\n", item["fine_tuned"])
